In [ ]:
!pip -q install pyspark

from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
import os

files.upload()

spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()

def read_csv(name):
    return spark.read.csv(name, header=True, inferSchema=True)

sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")

Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv


Part A - DataFrame operations

In [ ]:
sessions_df.printSchema()
print(sessions_df.count())
sessions_df.show(10)
for field in sessions_df.schema.fields:
    print(field.name, field.dataType)

root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+--------------

In [ ]:
selected_df = sessions_df.select(
    "session_id", "subscriber_id", "content_id",
    "watch_minutes", "session_status"
)
selected_df.filter(
    (F.col("session_status") == "Completed") &
    (F.col("watch_minutes") > 60)
).show()

+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|     SUB01859|    CT0075|          125|     Completed|
|WS00000179|     SUB00935|    CT0054|           62|     Completed|
|WS00000184|     SUB00072|    CT0119|           91|     Completed|
|WS00000222|     SUB01603|    CT0103|          116|     Completed|
|WS00000233|     SUB01233|    CT0027|          158|     Completed|
|WS00000244|     SUB00985|    CT0048|          140|     Completed|
|WS00000249|     SUB00681|    CT0100|          120|     Completed|
|WS00000257|     SUB02115|    CT0120|           68|     Completed|
|WS00000292|     SUB01068|    CT0048|          141|     Completed|
|WS00000326|     SUB00368|    CT0096|           97|     Completed|
|WS00000415|     SUB01758|    CT0105|          150|     Completed|
|WS00000419|     SUB00367|    CT0036|           99|     Comple

In [ ]:
engagement_df = sessions_df.withColumn(
    "watch_hours", F.col("watch_minutes") / 60
).withColumn(
    "high_engagement",
    F.when(F.col("completion_pct").isNull(), None)
     .when(F.col("completion_pct") >= 75, True)
     .otherwise(False)
)

engagement_df.show()

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+---------------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|        watch_hours|high_engagement|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+---------------+
|WS00000001|     SUB00157|    CT0097|     DV02|2026-04-26 18:59:00|            6|          11.8|             29.0|            HD|     Abandoned|        72.4|       NULL|                0.1|          false|
|WS00000002|     SUB01430|    CT0031|     DV04|2026-07-30 06:26:00|           31|          56.4|             31.0|            HD|   Interrupted|       422.7|       NULL| 0.5166

In [ ]:
sessions_df.orderBy(
    F.col("data_used_mb").desc()
).show(20)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|     SUB00155|    CT0069|     DV08|2026-07-20 00:11:00|          167|          97.7|              9.0|            4K|     Completed|      8881.8|        3.0|
|WS00074513|     SUB00684|    CT0056|     DV05|2026-08-09 21:53:00|          157|          96.9|              8.0|            4K|     Completed|      8408.4|       NULL|
|WS00039201|     SUB01664|    CT0027|     DV08|2026-05-16 06:35:00|          161|          97.0|              9.0|            4K|     Completed|      

Part B - Data quality and conditional transformations

In [ ]:
columns = [
    "subscriber_id", "device_id", "completion_pct",
    "buffering_seconds", "user_rating"
]
sessions_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in columns
]).show()

+-------------+---------+--------------+-----------------+-----------+
|subscriber_id|device_id|completion_pct|buffering_seconds|user_rating|
+-------------+---------+--------------+-----------------+-----------+
|          951|     2003|          3513|             4424|      77605|
+-------------+---------+--------------+-----------------+-----------+



In [ ]:
filled_df = sessions_df.fillna({
    "buffering_seconds": 0
})
filled_df.show(10)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000001|     SUB00157|    CT0097|     DV02|2026-04-26 18:59:00|            6|          11.8|             29.0|            HD|     Abandoned|        72.4|       NULL|
|WS00000002|     SUB01430|    CT0031|     DV04|2026-07-30 06:26:00|           31|          56.4|             31.0|            HD|   Interrupted|       422.7|       NULL|
|WS00000003|     SUB01892|    CT0044|     DV08|2026-02-22 20:55:00|           29|          63.0|             26.0|            HD|   Interrupted|      

In [ ]:
cleaned_df = sessions_df.dropna(subset=["device_id"])
print("Before:", sessions_df.count())
print("After:", cleaned_df.count())

Before: 80000
After: 77997


In [ ]:
band_df = sessions_df.withColumn(
    "completion_band",
    F.when(F.col("completion_pct").isNull(), "Unknown")
     .when(F.col("completion_pct") < 30, "Low")
     .when(F.col("completion_pct") < 90, "Medium")
     .otherwise("High")
)
band_df.select("completion_pct", "completion_band").show()

+--------------+---------------+
|completion_pct|completion_band|
+--------------+---------------+
|          11.8|            Low|
|          56.4|         Medium|
|          63.0|         Medium|
|          75.0|         Medium|
|          23.4|            Low|
|          19.8|            Low|
|          83.9|         Medium|
|          66.7|         Medium|
|          80.8|         Medium|
|          65.6|         Medium|
|          36.1|         Medium|
|          78.6|         Medium|
|          51.6|         Medium|
|          28.1|            Low|
|          16.4|            Low|
|          28.0|            Low|
|          84.3|         Medium|
|          83.3|         Medium|
|          90.3|           High|
|          27.0|            Low|
+--------------+---------------+
only showing top 20 rows


Part C - Joins and aggregations

In [ ]:
sessions_df.join(
    content_df, "content_id"
).groupBy("genre").agg(
    F.count("*").alias("session_count"),
    F.sum("watch_minutes").alias("total_watch_minutes")
).show()

+-----------+-------------+-------------------+
|      genre|session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|   Thriller|         5972|             248430|
|  Adventure|        15561|             839700|
|     Sports|         9967|             383044|
|      Drama|        10733|             519872|
|Documentary|         7432|             256626|
|  Animation|        11886|             490335|
|     Comedy|         9105|             452563|
|     Sci-Fi|         9344|             433946|
+-----------+-------------+-------------------+



In [ ]:
sessions_df.join(
    devices_df, "device_id"
).groupBy("device_type").agg(
    F.avg("buffering_seconds").alias("avg_buffering"),
    F.avg("watch_minutes").alias("avg_watch_minutes")
).show()

+---------------+------------------+------------------+
|    device_type|     avg_buffering| avg_watch_minutes|
+---------------+------------------+------------------+
|         iPhone|35.761106907361366|45.226752503576535|
|Streaming Stick|36.320860309448044| 45.08570541351827|
|Desktop Browser| 36.02889125799574|45.571818733562615|
|  Android Phone|35.676486693680864|  45.0885616509385|
| Laptop Browser|36.930938301907844| 45.84465713083719|
|         Tablet| 35.23614616481949| 45.38943894389439|
|       Smart TV| 36.67392739273927| 45.33776623376623|
|   Game Console| 35.51243726816496| 45.04469158686026|
+---------------+------------------+------------------+



In [ ]:
plan_sessions = sessions_df.filter(
    F.col("subscriber_id").isNotNull()
).join(
    subscribers_df, "subscriber_id"
).join(
    plans_df, "plan_id"
)
plan_sessions.groupBy("plan_name").agg(
    F.count("*").alias("total_sessions"),
    F.sum(
        F.when(F.col("session_status") == "Completed", 1)
         .otherwise(0)
    ).alias("completed_sessions")
).withColumn(
    "completion_rate",
    F.round(
        F.col("completed_sessions") * 100 /
        F.col("total_sessions"), 2
    )
).show()

+---------+--------------+------------------+---------------+
|plan_name|total_sessions|completed_sessions|completion_rate|
+---------+--------------+------------------+---------------+
|  Premium|         12198|               975|           7.99|
|    Basic|         25086|              2006|            8.0|
|   Mobile|         17510|              1429|           8.16|
| Standard|         23684|              1868|           7.89|
+---------+--------------+------------------+---------------+



In [ ]:
subscribers_df.dropDuplicates(
    ["subscriber_id"]
).filter(
    F.col("subscriber_status") == "Active"
).join(
    plans_df, "plan_id"
).groupBy("plan_name").agg(
    F.count("*").alias("active_subscribers"),
    F.sum("monthly_fee_inr").alias("monthly_revenue")
).show()

+---------+------------------+---------------+
|plan_name|active_subscribers|monthly_revenue|
+---------+------------------+---------------+
|  Premium|               326|         211574|
|    Basic|               632|         157368|
|   Mobile|               443|          66007|
| Standard|               604|         240996|
+---------+------------------+---------------+



In [ ]:
unmatched_subscribers = sessions_df.filter(
    F.col("subscriber_id").isNotNull()
).join(
    subscribers_df.select("subscriber_id").distinct(),
    "subscriber_id",
    "left_anti"
)
unmatched_devices = sessions_df.filter(
    F.col("device_id").isNotNull()
).join(
    devices_df.select("device_id").distinct(),
    "device_id",
    "left_anti"
)
print("Unmatched subscribers:", unmatched_subscribers.count())
print("Unmatched devices:", unmatched_devices.count())

Unmatched subscribers: 571
Unmatched devices: 481


Part D - Date and window functions

In [ ]:
date_df = sessions_df.withColumn(
    "session_start",F.to_timestamp("session_start")
)
date_df.filter(
    F.year("session_start") == 2026
).withColumn(
    "month", F.date_format("session_start", "yyyy-MM")
).groupBy("month").agg(
    F.count("*").alias("session_count"),
    F.sum("watch_minutes").alias("total_watch_minutes")
).orderBy("month").show()

+-------+-------------+-------------------+
|  month|session_count|total_watch_minutes|
+-------+-------------+-------------------+
|2026-01|         9070|             407215|
|2026-02|         8218|             372726|
|2026-03|         9019|             403971|
|2026-04|         8817|             399432|
|2026-05|         9006|             407386|
|2026-06|         8776|             404203|
|2026-07|         9204|             416104|
|2026-08|         8996|             407374|
|2026-09|         8894|             406105|
+-------+-------------+-------------------+



In [ ]:
signup_df = subscribers_df.withColumn(
    "signup_date", F.to_date("signup_date")
)
date_df.join(
    signup_df, "subscriber_id"
).withColumn(
    "days_since_signup",
    F.datediff(
        F.to_date("session_start"),
        F.col("signup_date")
    )
).orderBy("session_start").show(10)

+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-----------+------+-------+---------+-----------------+-----------------+
|subscriber_id|session_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|signup_date|region|plan_id|age_group|subscriber_status|days_since_signup|
+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-----------+------+-------+---------+-----------------+-----------------+
|     SUB01707|WS00035235|    CT0103|     DV07|2026-01-01 00:00:00|           60|          51.3|             11.0|            HD|   Interrupted|       819.7|       NULL| 2024-02-13|  East|   PL04|    18-24|           Active|              688|
|     SUB02206|WS00074739|  

In [ ]:
date_df.withColumn(
    "day_type",
    F.when(
        F.dayofweek("session_start").isin(1, 7),
        "Weekend"
    ).otherwise("Weekday")
).groupBy("day_type").agg(
    F.avg("watch_minutes").alias("avg_watch_minutes")
).show()

+--------+------------------+
|day_type| avg_watch_minutes|
+--------+------------------+
| Weekday| 45.26460823373174|
| Weekend|45.411601967328295|
+--------+------------------+



In [ ]:
title_df = sessions_df.join(
    content_df, "content_id"
).groupBy(
    "genre", "content_id", "content_title"
).agg(
    F.sum("watch_minutes").alias("total_watch_minutes")
)
w = Window.partitionBy("genre").orderBy(
    F.col("total_watch_minutes").desc()
)
title_df.withColumn(
    "rank", F.dense_rank().over(w)
).filter(
    F.col("rank") <= 3
).orderBy("genre", "rank").show(50)

+-----------+----------+----------------+-------------------+----+
|      genre|content_id|   content_title|total_watch_minutes|rank|
+-----------+----------+----------------+-------------------+----+
|  Adventure|    CT0105|Infinite Horizon|              66395|   1|
|  Adventure|    CT0048|    Lost Journey|              61823|   2|
|  Adventure|    CT0042|    Solar Memory|              60540|   3|
|  Animation|    CT0106|      Last River|              51398|   1|
|  Animation|    CT0114| Emerald Circuit|              51310|   2|
|  Animation|    CT0100|   Golden Puzzle|              50515|   3|
|     Comedy|    CT0027|    Blue Horizon|              61428|   1|
|     Comedy|    CT0019|   Bright Valley|              60921|   2|
|     Comedy|    CT0059|   Silver Valley|              60422|   3|
|Documentary|    CT0077|   Fading Signal|              47085|   1|
|Documentary|    CT0057|  Emerald Valley|              40819|   2|
|Documentary|    CT0014|   Solar Horizon|              36152| 

In [ ]:
w = Window.partitionBy("subscriber_id").orderBy(
    F.col("session_start").desc(),
    F.col("session_id").desc()
)
date_df.filter(
    F.col("subscriber_id").isNotNull()
).withColumn(
    "rn", F.row_number().over(w)
).filter(
    F.col("rn") == 1
).show()

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+---+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating| rn|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+---+
|WS00020822|     SUB00001|    CT0073|     DV03|2026-09-11 19:19:00|           34|          89.5|            145.0|            HD|   Interrupted|       418.3|       NULL|  1|
|WS00032744|     SUB00002|    CT0076|     DV05|2026-09-02 05:41:00|           45|          73.8|            219.0|       Full HD|   Interrupted|      1184.1|       NULL|  1|
|WS00006110|     SUB00003|    CT0004|     DV03|2026-09-30 03:13:00|           11|          15.3|              3.0|            4K| 

In [ ]:
w = Window.partitionBy("subscriber_id").orderBy(
    "session_start", "session_id"
)
date_df.filter(
    F.col("subscriber_id").isNotNull()
).withColumn(
    "previous_watch_minutes",
    F.lag("watch_minutes").over(w)
).withColumn(
    "difference",
    F.col("watch_minutes") - F.col("previous_watch_minutes")
).select(
    "subscriber_id", "session_id", "watch_minutes",
    "previous_watch_minutes", "difference"
).show()

+-------------+----------+-------------+----------------------+----------+
|subscriber_id|session_id|watch_minutes|previous_watch_minutes|difference|
+-------------+----------+-------------+----------------------+----------+
|     SUB00001|WS00064390|           58|                  NULL|      NULL|
|     SUB00001|WS00045183|           78|                    58|        20|
|     SUB00001|WS00028484|            4|                    78|       -74|
|     SUB00001|WS00043555|          108|                     4|       104|
|     SUB00001|WS00041862|           79|                   108|       -29|
|     SUB00001|WS00059616|           18|                    79|       -61|
|     SUB00001|WS00042345|           44|                    18|        26|
|     SUB00001|WS00009166|           91|                    44|        47|
|     SUB00001|WS00003744|           27|                    91|       -64|
|     SUB00001|WS00029676|           63|                    27|        36|
|     SUB00001|WS00043178

Part E - Union, lazy execution, caching and partitioning

In [ ]:
combined_df = sessions_df.unionByName(extra_df)
unique_df = combined_df.dropDuplicates(["session_id"])
print("Main:", sessions_df.count())
print("Extra:", extra_df.count())
print("Combined:", combined_df.count())
print("Unique:", unique_df.count())
print("Duplicates removed:", combined_df.count() - unique_df.count())

Main: 80000
Extra: 10000
Combined: 90000
Unique: 89400
Duplicates removed: 600


In [ ]:
pipeline_df = sessions_df.filter(
    F.col("session_status") == "Completed"
).join(
    content_df, "content_id"
).withColumn(
    "watch_hours", F.col("watch_minutes") / 60
).groupBy("genre").agg(
    F.sum("watch_hours").alias("total_hours")
).orderBy(
    F.col("total_hours").desc()
)
pipeline_df.explain()
pipeline_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_hours#1192 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_hours#1192 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2605]
      +- HashAggregate(keys=[genre#100], functions=[sum(watch_hours#1191)])
         +- Exchange hashpartitioning(genre#100, 200), ENSURE_REQUIREMENTS, [plan_id=2602]
            +- HashAggregate(keys=[genre#100], functions=[partial_sum(watch_hours#1191)])
               +- Project [genre#100, (cast(watch_minutes#22 as double) / 60.0) AS watch_hours#1191]
                  +- BroadcastHashJoin [content_id#19], [content_id#98], Inner, BuildRight, false
                     :- Project [content_id#19, watch_minutes#22]
                     :  +- Filter ((isnotnull(session_status#26) AND (session_status#26 = Completed)) AND isnotnull(content_id#19))
                     :     +- FileScan csv [content_id#19,watch_minutes#22,session_status#26] Batched: false, DataFilters: [isnot

In [ ]:
completed_df = sessions_df.filter(
    F.col("session_status") == "Completed"
)
completed_df.cache()
for i in range(2):
    start = time.time()
    completed_df.count()
    completed_df.groupBy("stream_quality").agg(
        F.avg("watch_minutes").alias("avg_watch")
    ).show()
    print("Run", i + 1, "Time:", time.time() - start)
print("Cached:", completed_df.is_cached)
completed_df.unpersist()

+--------------+-----------------+
|stream_quality|        avg_watch|
+--------------+-----------------+
|       Full HD|77.52547169811321|
|            4K|77.31385642737897|
|            SD|76.68812330009067|
|            HD|76.39961089494163|
+--------------+-----------------+

Run 1 Time: 1.2528181076049805
+--------------+-----------------+
|stream_quality|        avg_watch|
+--------------+-----------------+
|       Full HD|77.52547169811321|
|            4K|77.31385642737897|
|            SD|76.68812330009067|
|            HD|76.39961089494163|
+--------------+-----------------+

Run 2 Time: 0.2781186103820801
Cached: True


DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [ ]:
quality_df = sessions_df.filter(
    F.col("stream_quality") == "4K"
)
quality_df.persist(StorageLevel.MEMORY_AND_DISK)
print(quality_df.count())
quality_df.groupBy("session_status").count().show()
quality_df.unpersist()

7312
+--------------+-----+
|session_status|count|
+--------------+-----+
|     Completed|  599|
|     Abandoned| 1256|
|   Interrupted| 5457|
+--------------+-----+



DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [ ]:
df8 = sessions_df.repartition(8)
print("Partitions:", df8.rdd.getNumPartitions())
df8.withColumn(
    "partition_id", F.spark_partition_id()
).groupBy("partition_id").count().orderBy(
    "partition_id"
).show()

Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|10000|
|           1| 9999|
|           2|10001|
|           3|10000|
|           4|10000|
|           5|10000|
|           6|10000|
|           7|10000|
+------------+-----+



In [ ]:
coalesced_df = df8.coalesce(2)
repartitioned_df = df8.repartition(2)
coalesced_df.explain()
repartitioned_df.explain()
repartitioned_df.write.mode("overwrite").parquet(
    "streaming_output"
)
print(os.listdir("streaming_output"))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=3220]
      +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/watch_sessions_80000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<session_id:string,subscriber_id:string,content_id:string,device_id:string,session_start:ti...


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=3229]
   +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] Ba

Final capstone - Streaming performance report

In [ ]:
combined = sessions_df.unionByName(extra_df)
combined = combined.dropDuplicates(["session_id"])

combined = combined.withColumn(
    "session_start", F.to_timestamp("session_start")
).filter(
    F.col("session_start").isNotNull()
).fillna({
    "buffering_seconds": 0,
    "data_used_mb": 0
})
combined = combined.withColumn(
    "month", F.date_format("session_start", "yyyy-MM")
)

enriched = combined.join(
    subscribers_df, "subscriber_id", "left"
).join(
    content_df, "content_id", "left"
).join(
    devices_df, "device_id", "left"
).join(
    plans_df, "plan_id", "left"
)

enriched = enriched.withColumn(
    "watch_hours",
    F.col("watch_minutes") / 60
).withColumn(
    "completed_flag",
    F.when(F.col("session_status") == "Completed", 1)
     .otherwise(0)
)

report_df = enriched.groupBy(
    "region", "genre", "month"
).agg(
    F.count("*").alias("session_count"),
    F.sum("watch_hours").alias("total_watch_hours"),
    F.sum("completed_flag").alias("completed_sessions"),
    F.avg("buffering_seconds").alias("avg_buffering"),
    F.sum("data_used_mb").alias("total_data_mb")
).withColumn(
    "completion_rate",
    F.round(
        F.col("completed_sessions") * 100 /
        F.col("session_count"), 2
    )
)
report_df.show(20)

title_report = enriched.groupBy(
    "genre", "content_id", "content_title"
).agg(
    F.sum("watch_hours").alias("total_watch_hours")
)
w = Window.partitionBy("genre").orderBy(
    F.col("total_watch_hours").desc()
)

top_titles = title_report.withColumn(
    "rank", F.dense_rank().over(w)
).filter(
    F.col("rank") <= 3
)
top_titles.show(50)

report_df.cache()
print("Report count:", report_df.count())
report_df.orderBy(
    F.col("total_watch_hours").desc()
).show(20)
report_df.unpersist()

report_df.repartition(3).write.mode("overwrite").parquet(
    "streaming_performance_report"
)
print(os.listdir("streaming_performance_report"))

final_df = spark.read.parquet("streaming_performance_report")
final_df.show(20, truncate=False)

+-------+-----------+-------+-------------+------------------+------------------+------------------+------------------+---------------+
| region|      genre|  month|session_count| total_watch_hours|completed_sessions|     avg_buffering|     total_data_mb|completion_rate|
+-------+-----------+-------+-------------+------------------+------------------+------------------+------------------+---------------+
|   East|  Animation|2026-09|          206|             147.9|                21|33.300970873786405|          157941.0|          10.19|
|  North|Documentary|2026-04|          205|111.94999999999999|                14|28.921951219512195|124649.20000000001|           6.83|
|   West|     Sports|2026-06|          311|217.33333333333337|                30| 35.47909967845659|236408.30000000005|           9.65|
|   East|      Drama|2026-06|          173|153.36666666666665|                19|  34.2485549132948|160250.60000000003|          10.98|
|   West|Documentary|2026-04|          233|135.9